In [ ]:
# Geolibraries
import geopandas as gpd
import osmnx as ox
import contextily as ctx


# R5
import r5py
from r5py import TransportNetwork

# General tools
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

In [ ]:
users = pd.read_parquet("scratch/data/user_home+work.parquet")

In [ ]:
users = users[
    (users["YY"] == 2024) &
    (users["MM"].isin([3, 4, 5]))
].copy()

In [ ]:
stays = pd.read_parquet("scratch/data/stays_sufficient_users.parquet")

In [ ]:
stays

In [ ]:
import geopandas as gpd

# Load the study area polygon
gdf = gpd.read_file("./data/Turku_region_boundary.geojson")  # or .shp

# Make sure it’s in WGS84 (lat/lon) for OSMnx
gdf = gdf.to_crs(epsg=4326)

In [ ]:
import geopandas as gpd

# Load the study area polygon
gdf_tampere = gpd.read_file("./data/Tampere_region_boundary.geojson")  # or .shp

# Make sure it’s in WGS84 (lat/lon) for OSMnx
gdf_tampere = gdf_tampere.to_crs(epsg=4326)

In [ ]:
import geopandas as gpd

# Load the study area polygon
gdf_oulu = gpd.read_file("./data/oulu_region_boundary.geojson")  # or .shp

# Make sure it’s in WGS84 (lat/lon) for OSMnx
gdf_oulu = gdf_oulu.to_crs(epsg=4326)

In [ ]:
import h3

geom = gdf.geometry.iloc[0]

turku_hexes = set()

if geom.geom_type == "Polygon":
    coords = [(lat, lon) for lon, lat in geom.exterior.coords]
    turku_hexes.update(
        h3.polyfill({"type": "Polygon", "coordinates": [coords]}, res=9)
    )

elif geom.geom_type == "MultiPolygon":
    for poly in geom.geoms:
        coords = [(lat, lon) for lon, lat in poly.exterior.coords]
        turku_hexes.update(
            h3.polyfill({"type": "Polygon", "coordinates": [coords]}, res=9)
        )


In [ ]:
import h3

geom = gdf_tampere.geometry.iloc[0]

tampere_hexes = set()

if geom.geom_type == "Polygon":
    coords = [(lat, lon) for lon, lat in geom.exterior.coords]
    tampere_hexes.update(
        h3.polyfill({"type": "Polygon", "coordinates": [coords]}, res=9)
    )

elif geom.geom_type == "MultiPolygon":
    for poly in geom.geoms:
        coords = [(lat, lon) for lon, lat in poly.exterior.coords]
        tampere_hexes.update(
            h3.polyfill({"type": "Polygon", "coordinates": [coords]}, res=9)
        )


In [ ]:
import h3

geom = gdf_oulu.geometry.iloc[0]

oulu_hexes = set()

if geom.geom_type == "Polygon":
    coords = [(lat, lon) for lon, lat in geom.exterior.coords]
    oulu_hexes.update(
        h3.polyfill({"type": "Polygon", "coordinates": [coords]}, res=9)
    )

elif geom.geom_type == "MultiPolygon":
    for poly in geom.geoms:
        coords = [(lat, lon) for lon, lat in poly.exterior.coords]
        oulu_hexes.update(
            h3.polyfill({"type": "Polygon", "coordinates": [coords]}, res=9)
        )


In [ ]:
stays['in_turku'] = stays['stay_gid9'].isin(turku_hexes)

In [ ]:
stays['in_tampere'] = stays['stay_gid9'].isin(tampere_hexes)

In [ ]:
stays['in_oulu'] = stays['stay_gid9'].isin(oulu_hexes)

In [ ]:
stays = stays[stays["in_turku"] == True]

In [ ]:
stays = stays[stays["in_tampere"] == True]

In [ ]:
stays = stays[stays["in_oulu"] == True]

In [ ]:
stays = stays[
    (stays["YY"] == 2024) &
    (stays["MM"].isin([3, 4, 5]))
].copy()

In [ ]:
stays.sort_values(by="total_visit_frequency", ascending=False)

In [ ]:
frequency_period = (
    stays
    .groupby(["user_id", "stay_gid9"], as_index=False)
    .agg(
        frequency_period=("total_visit_frequency", "sum")
    )
)

In [ ]:
frequency_period

In [ ]:
stays_summary = (
    frequency_period
    .groupby("user_id")
    .agg(
        n_stays=("user_id", "count"),
        max_visit_frequency=("frequency_period", "max")
    )
    .reset_index()
)

In [ ]:
stays_summary.describe()

In [ ]:
plt.figure()
plt.hist(stays_summary["n_stays"], bins=30)
plt.xlabel("Number of stays per user")
plt.ylabel("Count")
plt.title("Distribution of number of stays per user")
plt.show()

In [ ]:
plt.figure()
plt.hist(stays_summary["max_visit_frequency"], bins=30)
plt.xlabel("Maximum visit frequency")
plt.ylabel("Count")
plt.title("Distribution of maximum visit frequency per user")
plt.show()

In [ ]:
plt.figure()
plt.hist(stays_summary["n_stays"], bins=100)
plt.yscale("log")
plt.xlabel("Number of stays per user")
plt.ylabel("Count (log scale)")
plt.title("Distribution of number of stays per user (log scale)")
plt.show()

In [ ]:
users['turku_home'] = users['home_gid9'].isin(turku_hexes)

### How xiuning defined in_home ?? maybe home_gid9 is 

In [ ]:
users['tampere_home'] = users['home_gid9'].isin(tampere_hexes)

In [ ]:
users['oulu_home'] = users['home_gid9'].isin(oulu_hexes)

In [ ]:
users_home = (
    users
    .loc[users["turku_home"] == True]
    .sort_values(["YY", "MM"])   # first year, then month
    .drop_duplicates(
        subset="user_id",
        keep="last"
    )[["user_id", "home_gid9", "work_gid9"]]
    .copy()
)

In [ ]:
users_home = (
    users
    .loc[users["tampere_home"] == True]
    .sort_values(["YY", "MM"])   # first year, then month
    .drop_duplicates(
        subset="user_id",
        keep="last"
    )[["user_id", "home_gid9", "work_gid9"]]
    .copy()
)

In [ ]:
users_home = (
    users
    .loc[users["oulu_home"] == True]
    .sort_values(["YY", "MM"])   # first year, then month
    .drop_duplicates(
        subset="user_id",
        keep="last"
    )[["user_id", "home_gid9", "work_gid9"]]
    .copy()
)

In [ ]:
users_home = users_home[users_home["work_gid9"].notna()].copy()
# Only those with a work_gid

In [ ]:

# 2. Merge with frequency_period
df_merged = frequency_period.merge(
    users_home,
    on="user_id",
    how="left"
)

In [ ]:
df_merged

In [ ]:
df_merged["is_home"] = (df_merged["stay_gid9"] == df_merged["home_gid9"]).astype(int)
df_merged["is_work"] = (df_merged["stay_gid9"] == df_merged["work_gid9"]).astype(int)

In [ ]:
df_merged.nunique()

In [ ]:
df_merged[df_merged["user_id"]=="fff81e73-8162-492e-9e7c-c9bed63db339"].sort_values("stay_gid9")

In [ ]:
df_merged.to_parquet("scratch/data/users_and_stays_3months_oulu.parquet")